In [143]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn import set_config
from sklearn.model_selection import train_test_split

from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# Random Surivial Forests

- an ensemble of survival trees
- survival = survival event + time object
- uses the log-rank test for feature & threshold selection

In [146]:
# load datasets
expr = pd.read_csv("../datasets/csv_files/expression_matrix_train.csv")
clinical = pd.read_csv("../datasets/csv_files/clinical_metadata_train.csv")
expr_test = pd.read_csv("../datasets/csv_files/clinical_metadata_test_one.csv")
clinical_test = pd.read_csv("../datasets/csv_files/clinical_metadata_test_one.csv")
clinical_test.head()

,Unnamed: 0,sample_id,is_tumor,patient_age,tumor_grade,t_stage,er_status,lymph_node_status,relapse_free_event,relapse_free_time
0,GSM540108,GSM540108,1,67.0,2.0,2.0,1.0,1.0,1.0,731.0
1,GSM540109,GSM540109,1,47.0,3.0,NaN,0.0,1.0,1.0,NaN
2,GSM540110,GSM540110,1,41.0,3.0,NaN,1.0,1.0,1.0,NaN
3,GSM540111,GSM540111,1,62.0,3.0,1.0,0.0,1.0,1.0,1770.0
4,GSM540112,GSM540112,1,60.0,2.0,2.0,1.0,1.0,1.0,871.0


In [145]:
# drop non-tumor samples from datasets
cols_to_keep = list(clinical[clinical["is_tumor"] == 1]['sample_name'])
expr_filtered = expr[['gene_symbol'] + cols_to_keep]
clinical_filtered = clinical[clinical["is_tumor"] == 1]

# drop non-tumor samples from datasets
cols_to_keep_test = list(clinical_test[clinical_test["is_tumor"] == 1]['sample_name'])
expr_test_filtered = expr_test[['gene_symbol'] + cols_to_keep_test]
clinical_test_filtered = clinical_test[clinical_test["is_tumor"] == 1]
expr_test_filtered.head()

KeyError: 'sample_name'

In [ ]:
expr_t = expr_filtered.set_index('gene_symbol').T
expr_t.index.name = 'sample_name'
expr_t.reset_index(inplace=True)
expr_t.columns.name = None
expr_t.head()

expr_test_t = expr_test_filtered.set_index('gene_symbol').T
expr_test_t.index.name = 'sample_name'
expr_test_t.reset_index(inplace=True)
expr_test_t.columns.name = None

,sample_name,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,A4GNT,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
0,GSM1045208,5.423835,4.606007,3.249699,6.441683,3.656200,3.287500,3.647675,5.699903,3.211635,...,9.072691,5.848763,3.772115,5.328634,4.264500,5.138357,6.370620,4.791065,6.226584,3.038356
1,GSM1045209,6.177836,4.750312,3.358364,6.698725,3.981705,3.200001,3.350733,6.123236,3.351150,...,7.741966,4.962975,3.728532,5.500853,4.592497,4.587907,6.754799,4.861246,6.123540,3.009947
2,GSM1045210,6.000779,4.544140,3.748739,6.467165,3.977325,3.245106,3.625951,5.776333,3.439412,...,8.419945,5.801849,4.027637,5.198435,4.343625,4.800956,6.641229,4.789060,5.801374,3.209502
3,GSM1045211,6.153725,4.499905,3.697323,6.700450,4.022336,3.785318,4.531829,6.324131,3.696489,...,8.334740,4.707491,3.038488,4.903613,4.380842,4.640250,6.240870,5.034151,5.464579,3.075786
4,GSM1045212,5.198928,4.402814,3.152932,6.260376,3.764769,4.418914,3.285910,5.797102,3.288822,...,8.750623,5.483120,3.808385,5.407679,6.497051,5.116050,7.305620,5.051753,5.905793,3.078403


In [ ]:
# add relapse free survival time & event to training data
gene_cols = [col for col in expr_t.columns if col != 'sample_name']

train_data = pd.concat([
        expr_t['sample_name'],
        clinical_filtered[['relapse_free_time', 'relapse_free_event']].astype(int),
        expr_t[gene_cols]
    ], axis=1)

# drop null RFS values & rename columns
train_data.dropna(inplace=True)
train_data[['relapse_free_time', 'relapse_free_event']] = train_data[['relapse_free_time', 'relapse_free_event']].astype(int)

# get gene names
gene_names = list(train_data.columns[3:])

train_data.head()

,sample_name,relapse_free_time,relapse_free_event,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,...,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3,mir-223
17,GSM1045225,3026,0,5.006660,4.570530,3.052693,6.212599,4.311186,4.477361,3.720108,...,9.464618,5.080529,3.612147,5.422328,4.565276,5.209277,6.541682,4.941143,6.349697,3.069378
18,GSM1045226,755,1,6.308156,4.849502,3.276010,6.465102,4.034385,3.062556,3.349777,...,7.175808,5.788256,4.008946,5.313596,4.811102,5.625363,5.983060,4.981460,6.539641,2.964171
19,GSM1045227,3014,0,5.043475,4.543174,3.207718,6.398232,3.885093,3.310767,3.272844,...,10.697678,5.242314,3.898463,4.903048,6.729129,5.837191,6.982218,4.759069,6.022257,3.098914
20,GSM1045228,406,1,5.947000,4.793444,3.234156,6.255112,3.975388,3.109538,3.371143,...,8.163624,5.641581,3.613630,5.599734,6.467224,5.726108,5.411201,4.778602,6.210298,2.752077
21,GSM1045229,2225,0,5.389865,4.501806,3.094015,7.015752,3.892831,3.218178,3.337067,...,8.392371,4.979070,3.746636,5.230768,5.193031,5.544478,7.087539,5.130468,6.036339,2.870920


## Training

In [ ]:
X_train = train_data[gene_names]
y_train = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', train_data)
random_state = 20

In [ ]:
rsf = RandomSurvivalForest(n_estimators=1000, min_samples_split=10, min_samples_leaf=15, n_jobs=-1, random_state=random_state)
rsf.fit(X_train, y_train)

,n_estimators,1000
,max_depth,None
,min_samples_split,10
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,bootstrap,True
,oob_score,False
,n_jobs,-1
,random_state,20


## Testing

In [ ]:
# add relapse free survival time & event to testing data
gene_cols = [col for col in expr_t.columns if col != 'sample_name']

test_data = pd.concat([
        expr_test_t['sample_name'],
        clinical_test_filtered[['relapse_free_time', 'relapse_free_event']].astype(int),
        expr_test_t[gene_cols]
    ], axis=1)

# drop null RFS values & rename columns
test_data.dropna(inplace=True)
test_data[['relapse_free_time', 'relapse_free_event']] = train_data[['relapse_free_time', 'relapse_free_event']].astype(int)

# get gene names
gene_names = list(train_data.columns[3:])

train_data.head()

In [ ]:
X_test = test_data[gene_names]
y_test = Surv.from_dataframe('relapse_free_event', 'relapse_free_time', test_data)

In [ ]:
# c_index = rsf.score(X_test, y_test)
# f"{c_index:.5f}"

NameError: name 'X_test' is not defined